In [1]:
from pysr import PySRRegressor
import numpy as np
import torch
from torchmetrics.regression import R2Score

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [35]:
model = PySRRegressor.from_file(run_directory="outputs/20251217_133704_vqiCFf")

Attempting to load model from outputs/20251217_133704_vqiCFf/checkpoint.pkl...


In [36]:
model.model_selection = 'accuracy'

In [37]:
# --- prepare data ---
#data_train = np.load('dZdt_data/ConvNN+AE+D/20251215092753_train_n_features=7_latent_dims=3_past_timesteps=20_global_ocean_land_rp=False_tp=20_wd=0.0.npy')
data_test = np.load('dZdt_data/ConvNN+AE+D/20251215092753_val_n_features=7_latent_dims=3_past_timesteps=20_global_ocean_land_rp=False_tp=20_wd=0.0.npy')
latent_dims = 3
coord_skipper = None
n_features = 7
first_X_index = 3
subset_size = 10_000

def get_C_Pr_XZ_and_dZdt(data, latent_dims, coord_skipper):
    C = data[::coord_skipper, :, 0]
    Pr = data[::coord_skipper, :, 1: first_X_index]
    XZ = data[::coord_skipper, :, first_X_index: first_X_index + n_features + latent_dims]
    dZdt = data[::coord_skipper, :, first_X_index + n_features + latent_dims:]
    # Test if last shape of dZdt matches latent_dims
    assert dZdt.shape[2] == latent_dims, f"dZdt shape {dZdt.shape} does not match latent_dims {latent_dims}"
    feature_names = ['qv2m', 'Prw', 'T2m', 'Ts', 'Fs', 'Fl', 'l']
    for i in range(latent_dims):
        feature_names.append(f'z{i+1}')

    return C, Pr, XZ, dZdt, feature_names

C_test, Pr_test, XZ_test, dZdt_test, _ = get_C_Pr_XZ_and_dZdt(data_test, latent_dims, coord_skipper)

XZ_test = XZ_test.reshape(-1, n_features + latent_dims)
dZdt_test = dZdt_test.reshape(-1, latent_dims)

In [38]:
XZ_test.shape

(30310400, 10)

In [39]:
dZdt_pred = model.predict(XZ_test)

In [44]:
# Compute l2 norm between dZdt_test and dZdt_pred
l2_norm = np.linalg.norm(dZdt_test[:, 0] - dZdt_pred) / np.linalg.norm(dZdt_test[:, 2])
print(f"L2 norm between true and predicted dZdt: {l2_norm}")

L2 norm between true and predicted dZdt: 2.608815144551937


In [45]:
R2Score()(torch.tensor(dZdt_pred), torch.tensor(dZdt_test[:, 2]))

tensor(0.2050, dtype=torch.float64)